In [5]:
import numpy as np
import numpy as np
from scipy.stats import norm, uniform
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.optimize import bisect
from scipy.stats import Mixture, Normal, multivariate_normal
from scipy.optimize import brentq, bisect

In [11]:
rng = np.random.default_rng(seed = 10404)
def cl(ndim, sigma = 1.0):
    '''
    Common Laguage Definition: Create common language covariance matrix(ndim,ndim)
    
    Ndim: Number of dimensions of  common language covariance matrix
    Sigma: Standard deviation of common language covariance matrix entries.
    Returns:
    Mixture: Common language normal mixture with mean N(0,sigma) where sigma is diaganol of common language matrix.
    Common_language_mat: Returns common_language covariance matrix (ndim, ndim)
    

    '''
    A = rng.normal(0,sigma, size = (ndim,ndim))
    common_language_mat = A @ A.T
    variances = np.diag(common_language_mat)
    components = [Normal(mu=0, sigma = np.sqrt(v)) for v in variances]
    return common_language_mat, Mixture(components)
def dialect_matrices(common_language, n_dialect, sigma=.01, mu_scale = .01):
    '''
    dialect_matrices:
    Generates n dialect Gram matrices with small pertubations induced by random normal mean. 

    common_language: Output of cl function, common language covariance matrix (ndim,ndim)
    n_dialect: number of dialect matrices to be created : int
    simga: scale of rng.normal draw used for entries of dialect matrix: float
    mu_scale: scale of rng.normal draw that governs individual dialect matrix means

    '''

    mat = []

    ndim = common_language.shape[0]

    for _ in range(n_dialect):
        mu = rng.normal(0, scale=mu_scale)
        P = rng.normal(mu, scale=sigma, size=(ndim, ndim))
        X = P @ P.T
        mat.append(X)
    return mat

  
def vector_sampling(common_language, dialect_matrices, n_samples=1000):
    '''
    vector_sampling: Take n vector samples from common_language matrix and n vector samples from random dialect matrices and sums them. 


    common_language: (ndim,ndim) common language matrix returned by cl
    dialect_matrices: (ndim,ndim) array of dialect matrices returned by dialect_matrices
    n_sample: Number of vectors samples to be taken
    Returns:
    Common_vectors: Randomly sampled vectors from common language matrix
    Perturbed_vectors: Sum of randomly sampled common language vectors and dialect vectors
    '''
    n_dim = common_language.shape[0]
    K = len(dialect_matrices)
    d_stack = np.stack(dialect_matrices)  

    common_inds = rng.integers(0, n_dim, size=n_samples)
    mat_inds    = rng.integers(0, K,     size=n_samples)
    d_inds      = rng.integers(0, n_dim, size=n_samples)

    common_vectors    = common_language[common_inds]              
    dialect_vectors   = d_stack[mat_inds, d_inds]                 
    perturbed_vectors = common_vectors + dialect_vectors

    return common_vectors, perturbed_vectors


  









In [17]:
def tokenize_vectors(common_vectors, perturbed_vectors,common_language_mixture, n_bins=10000, eps=1e-7):
    '''
    tokenize_vectors: Creates n_bins across token space defined by boundaries given by inverse cumulative function of normal mixture defined in cl()
    common_vectors: Common language vector output from vector_sampling()
    perturbed_vectors: Perturbed language vector output from vector_sampling()
    common_language_mixture: Normal mixture object from Scipy.stats defined in cl()
    n_bins: number of bins for token space: int
    eps: Probabalistic boundary of tokenizable space, used with inverse cumulative function of normal mixture
    Returns:
    common_tokens: Tokens across n bins (0, n_bins) created from un-perturbed common language vectors
    perturbed_tokens: Tokens across n bins(0,bins) created from perturbed language vectors.
    '''
    lo = common_language_mixture.icdf(eps)
    hi = common_language_mixture.icdf(1 - eps)
    bins = np.linspace(lo, hi, n_bins + 1)

    def apply_tokens(vectors):
        tokens = np.digitize(vectors.ravel(), bins).astype(object)
        tokens[(tokens == 0) | (tokens == n_bins + 1)] = "unknown"
        return tokens.reshape(vectors.shape)

    common_tokens    = apply_tokens(common_vectors)
    perturbed_tokens = apply_tokens(perturbed_vectors)
    return common_tokens, perturbed_tokens

In [8]:
def match_rate(common_tokens, perturbed_tokens):
    '''
    match_rate: Compares tokenization of common_vectors and perturbed vectors 
    common_tokens
    perturbed_tokens
    returns:
    match_rate: percentage of tokenization shared between both common and perturbed vectors. 
    '''
    match_rate = (common_tokens == perturbed_tokens).mean()
    return match_rate


In [19]:
'''
Sanity check, where sigma and mu_scale are 0, tokenizations should be the same and match_rate should be 1
'''
mat, mix = cl(30)
dm = dialect_matrices(mat, 30, 0,0)
c_v , p_v = vector_sampling(mat,dm,100)
c_t , p_t = tokenize_vectors(c_v, p_v, mix,n_bins = 1000, eps = 1e-7)




np.float64(1.0)